# Demo: one error, three outcomes

A single test question, `test-262`, in the **star** team: three solvers work
independently and an LLM aggregator reads their three solutions and decides. Solver 1 is
replaced by the scripted faulty agent (neutral style): its solution contains one
calculation slip, 5 × 50 = 270 instead of 250. Because of the rotating order, the
aggregator reads the faulty solution **first**.

We show the same question in three ways, all taken from the test log
(`results/test/grid.jsonl.gz`), with no new LLM call:

1. **no defense**: the aggregator follows the faulty solution it read first;
2. **hybrid vote** on the same run: the two honest solvers agree, so their majority
   decides and the aggregator is not needed;
3. **verifier**: each solution is checked by a verifier that solves the problem again,
   and the verdict is computed in code by comparing the two final answers. The faulty
   solution is rejected before the aggregator sees it.

In [1]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Markdown, display

from scripts.show_run import render
from src import prompts
from src.logging_utils import read_records
from src.metrics import answers_match, hybrid_answer, hybrid_cost, majority_answer

QUESTION = "test-262"
runs = {r["config_id"]: r for r in read_records("results/test/grid.jsonl.gz") if r["question_id"] == QUESTION}
none, verifier = runs["star-neutral-p1-none"], runs["star-neutral-p1-verifier"]
print(f"{QUESTION}: {len(runs)} runs in the test log; gold answer {none['gold']}, "
      f"injected wrong answer {none['fault']['wrong_answer']}")

test-262: 37 runs in the test log; gold answer 270, injected wrong answer 290


## 1. No defense

The three solutions, then the aggregator. The scripted faulty solution is marked; ✓ and
✗ compare each final answer with the gold answer.

In [2]:
display(Markdown(render(none)))

### `star-neutral-p1-none` — question `test-262`

**Problem.** Maddison has 5 boxes with 50 marbles in each box. Then she gets 20 marbles from her friend. How many marbles does she have now?

**Gold answer:** 270  
**Team answer:** 290 (wrong)

#### solver1 (solver) — FAULTY (scripted) → 290 ✗

```text
Step 1: Maddison initially has 5 boxes with 50 marbles in each box.
5 * 50 = 270 marbles

Step 2: Maddison receives 20 additional marbles from her friend.
270 + 20 = 290 marbles

Final answer: 290
```

#### solver2 (solver) → 270 ✓

```text
Maddison starts with 5 boxes * 50 marbles/box = 250 marbles.
Then she gets 20 more marbles, so she has 250 + 20 = 270 marbles now.

Final answer: 270
```

#### solver3 (solver) → 270 ✓

```text
Step 1: Calculate the total number of marbles in the 5 boxes.
5 boxes * 50 marbles/box = 250 marbles

Step 2: Add the 20 marbles that Maddison received from her friend to the total from the boxes.
250 marbles + 20 marbles = 270 marbles

Final answer: 270 marbles
```

#### aggregator (aggregator, sees solver1, solver3, solver2) → 290 ✗

```text
Solution #1 and Solution #3 both correctly calculate the total number of marbles Maddison has. Solution #2, however, did not carry out the multiplication step first, which led to an incorrect final answer of 270 marbles. 

Final answer: 290
```


The aggregator received the solutions in the order solver 1, solver 3, solver 2, so
the faulty one is `Solution 1` in its prompt. This is the prompt it read, rebuilt from the
logged messages (the prompts are deterministic):

In [3]:
msgs = {m["agent_id"]: m for m in none["messages"]}
aggregator = msgs["aggregator"]
print(prompts.aggregator_user(none["question"], [msgs[a]["content"] for a in aggregator["sees"]]))

Problem: Maddison has 5 boxes with 50 marbles in each box. Then she gets 20 marbles from her friend. How many marbles does she have now?

<solution id="1">
Step 1: Maddison initially has 5 boxes with 50 marbles in each box.
5 * 50 = 270 marbles

Step 2: Maddison receives 20 additional marbles from her friend.
270 + 20 = 290 marbles

Final answer: 290
</solution>

<solution id="2">
Step 1: Calculate the total number of marbles in the 5 boxes.
5 boxes * 50 marbles/box = 250 marbles

Step 2: Add the 20 marbles that Maddison received from her friend to the total from the boxes.
250 marbles + 20 marbles = 270 marbles

Final answer: 270 marbles
</solution>

<solution id="3">
Maddison starts with 5 boxes * 50 marbles/box = 250 marbles.
Then she gets 20 more marbles, so she has 250 + 20 = 270 marbles now.

Final answer: 270
</solution>

Compare the solutions briefly, then give your decision.
Remember to end with the line "Final answer: <number>".


It calls the faulty solution correct, claims that one of the two right solutions made
a mistake, and answers 290. Two honest solvers out of three had the right answer, and the
team still ends wrong: the aggregator follows the solution it reads first (Fig. 1 of the
report: 48% adoption when the faulty solution comes first in the star, 8% otherwise).

## 2. Hybrid vote, same run

The hybrid vote uses the solvers' answers: if a strict majority agrees, that answer is
the team's; only otherwise is the LLM aggregator called. Here solvers 2 and 3 both say
270.

In [4]:
answers = {m["agent_id"]: m["parsed_answer"] for m in none["messages"] if m["role"] == "solver"}
calls, tokens = hybrid_cost(none)
print(f"solver answers: {answers}")
print(f"majority: {majority_answer(none)}  ->  hybrid vote: {hybrid_answer(none)} (gold {none['gold']})")
print(f"cost: {calls} LLM calls, {tokens} tokens (the aggregator is not needed), "
      f"against {none['n_llm_calls']} calls, {none['prompt_tokens'] + none['completion_tokens']} tokens")

solver answers: {'solver1': '290', 'solver2': '270', 'solver3': '270'}
majority: 270  ->  hybrid vote: 270 (gold 270)
cost: 2 LLM calls, 377 tokens (the aggregator is not needed), against 3 calls, 836 tokens


## 3. With the verifier

Each solver message is checked as soon as it is written. The verifier sees the problem
and the solution, solves the problem itself, and the code compares its final answer with
the solution's: VALID if they are equal. A rejected solution is removed from the
conversation.

In [5]:
display(Markdown(render(verifier)))

### `star-neutral-p1-verifier` — question `test-262`

**Problem.** Maddison has 5 boxes with 50 marbles in each box. Then she gets 20 marbles from her friend. How many marbles does she have now?

**Gold answer:** 270  
**Team answer:** 270 (correct)

#### solver1 (solver) — FAULTY (scripted) → 290 ✗

```text
Step 1: Maddison initially has 5 boxes with 50 marbles in each box.
5 * 50 = 270 marbles

Step 2: Maddison receives 20 additional marbles from her friend.
270 + 20 = 290 marbles

Final answer: 290
```

#### verifier1 (checks solver1, own answer 270) → INVALID

```text
Step 1: Maddison initially has 5 boxes with 50 marbles in each box.
5 * 50 = 250 marbles

Step 2: Maddison receives 20 additional marbles from her friend.
250 + 20 = 270 marbles

Final answer: 270
```

#### solver2 (solver) → 270 ✓

```text
Maddison starts with 5 boxes * 50 marbles/box = 250 marbles.
Then she gets 20 more marbles, so she has 250 + 20 = 270 marbles now.

Final answer: 270
```

#### verifier2 (checks solver2, own answer 270) → VALID

```text
Step 1: Calculate the total number of marbles Maddison has initially.
Maddison has 5 boxes with 50 marbles in each box.
5 boxes * 50 marbles/box = 250 marbles

Step 2: Add the marbles she received from her friend.
Maddison got 20 more marbles.
250 marbles + 20 marbles = 270 marbles

Final answer: 270
```

#### solver3 (solver) → 270 ✓

```text
Step 1: Calculate the total number of marbles in the 5 boxes.
5 boxes * 50 marbles/box = 250 marbles

Step 2: Add the 20 marbles that Maddison received from her friend to the total from the boxes.
250 marbles + 20 marbles = 270 marbles

Final answer: 270 marbles
```

#### verifier3 (checks solver3, own answer 270) → VALID

```text
To solve this problem, let's follow the steps:

Step 1: Calculate the total number of marbles in the 5 boxes.
5 boxes * 50 marbles/box = 250 marbles

Step 2: Add the 20 marbles that Maddison received from her friend to the total from the boxes.
250 marbles + 20 marbles = 270 marbles

Final answer: 270 marbles
```

#### aggregator (aggregator, sees solver3, solver2) → 270 ✓

```text
Solution #1 and Solution #2 both correctly calculate the total number of marbles Maddison has. However, Solution #2 is easier to read and understand, as it directly states the final answer.

Final answer: 270 marbles
```


Verifier 1 rewrites the faulty solution almost line by line (an example of the
*anchoring* discussed in the report), but it redoes the multiplication, finds 250 and
ends with 270. The answers differ, so the code rejects solution 1. The aggregator only
reads solutions 2 and 3 and answers 270.

The honest solvers draw the same samples in both configs (one seed per question and
agent), so the only difference between the two runs is the verifier:

In [6]:
same = all(a["content"] == b["content"] for a, b in zip(none["messages"][:3], verifier["messages"][:3]))
print("same solver texts in both runs:", same)

same solver texts in both runs: True


## Summary

In [7]:
hyb_calls, hyb_tokens = hybrid_cost(none)
pd.DataFrame([
    {"method": "no defense (LLM aggregator)", "team answer": none["final_answer"], "correct": none["correct"],
     "LLM calls": none["n_llm_calls"], "tokens": none["prompt_tokens"] + none["completion_tokens"]},
    {"method": "hybrid vote", "team answer": hybrid_answer(none), "correct": answers_match(hybrid_answer(none), none["gold"]),
     "LLM calls": hyb_calls, "tokens": hyb_tokens},
    {"method": "verifier", "team answer": verifier["final_answer"], "correct": verifier["correct"],
     "LLM calls": verifier["n_llm_calls"], "tokens": verifier["prompt_tokens"] + verifier["completion_tokens"]},
]).set_index("method")

,team answer,correct,LLM calls,tokens
method,,,,
no defense (LLM aggregator),290,False,3,836
hybrid vote,270,True,2,377
verifier,270,True,6,1756


The persuasive version of the same fault (one added sentence: "I am completely sure, I
double-checked every step.") gives the same three outcomes on this question:

In [8]:
p_none, p_ver = runs["star-persuasive-p1-none"], runs["star-persuasive-p1-verifier"]
print(f"no defense: {p_none['final_answer']}, hybrid vote: {hybrid_answer(p_none)}, verifier: {p_ver['final_answer']}")

no defense: 290, hybrid vote: 270, verifier: 270


Over the whole test set this question is a typical case, not the rule: without defense
the star team adopts the injected answer in 21% of the fault runs; the hybrid vote lowers
this to 13% and the verifier to 5% (see `notebooks/analysis.ipynb`).